In [16]:
!pip install selenium pyarrow lxml requests beautifulsoup4 pandas
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import os
import glob
import pickle
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.edge.options import Options

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36'
}

YEAR = 2022
SAVE_DIR = rf'C:\\Users\\lcsse\\Desktop\\STAGE_GENT\\scrapping\\data\\pcs{YEAR}'
os.makedirs(SAVE_DIR, exist_ok=True)

dates = []
list_of_result_dfs = []
races = []
data_by_month = {}


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [17]:
def is_page_not_found(response_text):
    return "Page not found" in response_text

def process_results_page(response_text, url):
    soup = BeautifulSoup(response_text, 'lxml')

    # Date
    try:
        date_tag = soup.select_one('div.value')
        date = date_tag.get_text(strip=True)
    except Exception:
        date = None
    dates.append(date)

    # Classification (2.UWT, 1.UWT, etc.)
    try:
        classif = None
        for li in soup.find_all('li'):
            if 'Classification:' in li.get_text():
                classif = li.get_text(strip=True).replace('Classification:', '').strip()
                break
    except Exception:
        classif = None

    # Startlist quality score
    try:
        startlist_quality = None
        for li in soup.find_all('li'):
            text = li.get_text()
            if 'Startlist quality score:' in text:
                match = re.search(r'Startlist quality score:\s*(\d+)', text)
                if match:
                    startlist_quality = int(match.group(1))
                break
    except Exception:
        startlist_quality = None

    # Average temperature
    try:
        avg_temperature = None
        for li in soup.find_all('li'):
            text = li.get_text()
            if 'Avg. temperature:' in text:
                match = re.search(r'Avg\. temperature:\s*([\d.]+)', text)
                if match:
                    avg_temperature = float(match.group(1))
                break
    except Exception:
        avg_temperature = None

    # Won how
    try:
        won_how = None
        for li in soup.find_all('li'):
            text = li.get_text()
            if 'Won how:' in text:
                won_how = text.split('Won how:')[-1].strip()
                break
    except Exception:
        won_how = None

    tables = soup.find_all('table')
    if tables:
        dfs = pd.read_html(str(tables))
        if dfs:
            df = dfs[0]
            df['date'] = date
            df['classification'] = classif
            df['startlist_quality'] = startlist_quality
            df['avg_temperature'] = avg_temperature
            df['won_how'] = won_how
            list_of_result_dfs.append(df)
            races.append(url)

def scrape_base_url(base_url):
    def has_results_table(response_text):
        soup = BeautifulSoup(response_text, 'lxml')
        return len(soup.find_all('table')) > 0

    result_url = base_url + "/result"
    response = requests.get(result_url, headers=headers)
    time.sleep(1)
    if has_results_table(response.text):
        process_results_page(response.text, result_url)

    stage = 1
    while True:
        stage_url = f"{base_url}/stage-{stage}"
        response = requests.get(stage_url, headers=headers)
        time.sleep(1)
        if has_results_table(response.text):
            process_results_page(response.text, stage_url)
            stage += 1
        else:
            break

    print(f"✅ {base_url}")

def scrape_urls(url_list):
    for base_url in url_list:
        scrape_base_url(base_url)

def clean_race_url(url):
    url = url.replace('/result', '')
    url = re.sub(r'/stage-\d+', '', url)
    return url

def save_month(month):
    for race, df, date in zip(data_by_month[month]['races'], data_by_month[month]['dfs'], data_by_month[month]['dates']):
        try:
            df['race_url'] = race
            df['date'] = date

            if '/stage-' in race:
                race_type = 'stage_' + race.split('/stage-')[-1]
            elif '/result' in race:
                race_type = 'result'
            else:
                race_type = 'one_day'

            race_name = race.replace('https://www.procyclingstats.com/race/', '')
            race_name = re.sub(r'/stage-\d+', '', race_name)
            race_name = race_name.replace('/result', '').replace('/', '_')

            clean_date = date.replace(' ', '_') if date else 'no_date'

            if 'classification' in df.columns and df['classification'].iloc[0]:
                classif = str(df['classification'].iloc[0]).strip()
            else:
                classif = 'no_classif'

            filename = f"{clean_date}__{race_type}__{race_name}__{classif}.csv"
            df.to_csv(os.path.join(SAVE_DIR, filename), index=False)
        except Exception as e:
            print(f"❌ {race}: {e}")
    print(f"✅ Mois {month}: {len(data_by_month[month]['races'])} fichiers sauvegardés")

In [18]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2022&month=1&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour janvier 2022")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[1] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(1)
driver.quit()

✅ 19 URLs trouvées pour janvier 2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and wi

✅ https://www.procyclingstats.com/race/new-zealand-cycle-classic/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-australia-itt/2022


...

✅ https://www.procyclingstats.com/race/gp-d-ouverture/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/grand-prix-megasaray/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/trofeo-palma/2022
Total brut: 45 | Uniques: 19 | Manquantes: 0
❌ https://www.procyclingstats.com/race/la-tropicale-amissa-bongo/2022/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/la-tropicale-amissa-bongo/2022/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/la-tropicale-amissa-bongo/2022/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/la-tropicale-amissa-bongo/2022/stage-4: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/la-tropicale-amissa-bongo/2022/stage-5: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/la-tropicale-amissa-bongo/2022/stage-6: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/la-tropicale-amissa-bongo/2022/stage-7: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-

In [19]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2022&month=2&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour fevrier 2022")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[2] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(2)
driver.quit()

✅ 35 URLs trouvées pour fevrier 2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and wi

✅ https://www.procyclingstats.com/race/alula-tour/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and wi

✅ https://www.procyclingstats.com/race/etoile-de-besseges/2022


...

✅ https://www.procyclingstats.com/race/faun-ardeche-classic/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/kuurne-brussel-kuurne/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/la-drome-classic/2022
Total brut: 79 | Uniques: 34 | Manquantes: 1
❌ https://www.procyclingstats.com/race/pas-cycling-road-race/2022/result: single positional indexer is out-of-bounds
✅ Mois 2: 79 fichiers sauvegardés


In [20]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2022&month=3&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour mars 2022")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[3] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(3)
driver.quit()

✅ 62 URLs trouvées pour mars 2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/gp-samyn/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/trofej-umag/2022


...

✅ https://www.procyclingstats.com/race/nc-bolivia/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/asian-championships-me/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/dwars-door-vlaanderen/2022
Total brut: 106 | Uniques: 61 | Manquantes: 1
❌ https://www.procyclingstats.com/race/liberation-of-kuwait-1/2022/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/ster-van-zwolle/2022/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/liberation-of-kuwait-2/2022/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-hainan/2022/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-hainan/2022/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-hainan/2022/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-hainan/2022/stage-4: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-hainan/2022/stage-5: single positional indexer is 

In [21]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2022&month=4&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour avril 2022")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[4] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(4)
driver.quit()

✅ 42 URLs trouvées pour avril 2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/route-adelie-de-vitre/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and wi

✅ https://www.procyclingstats.com/race/tour-of-thailand/2022


...

✅ https://www.procyclingstats.com/race/vuelta-asturias/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/zodc-zuidenveld-tour/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/grand-prix-wyszkow/2022
Total brut: 101 | Uniques: 42 | Manquantes: 0
❌ https://www.procyclingstats.com/race/adriatic-lustica-bay-montenegro-race/2022/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/adriatic-lustica-bay-montenegro-race/2022/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/klasika-primavera/2022/result: single positional indexer is out-of-bounds
✅ Mois 4: 101 fichiers sauvegardés


In [22]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2022&month=5&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour mai 2022")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[5] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(5)
driver.quit()

✅ 50 URLs trouvées pour mai 2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/eschborn-frankfurt/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/circuito-del-porto-trofeo-arvedi/2022


...

✅ https://www.procyclingstats.com/race/kyiv-speed-challenge/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/mercan-tour-classic-alpes-maritimes/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and wi

✅ https://www.procyclingstats.com/race/tour-of-albania/2022
Total brut: 128 | Uniques: 50 | Manquantes: 0
❌ https://www.procyclingstats.com/race/szlakiem-grodow-piastowskich/2022/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/szlakiem-grodow-piastowskich/2022/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/szlakiem-grodow-piastowskich/2022/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-de-lviv-region/2022/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-de-lviv-region/2022/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-de-lviv-region/2022/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-de-lviv-region/2022/stage-4: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-du-mali/202

In [23]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2022&month=6&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour juin 2022")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[6] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(6)
driver.quit()

✅ 178 URLs trouvées pour juin 2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/trofeo-alcide-degasperi/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/giro-dell-appennino/2022


...

✅ https://www.procyclingstats.com/race/nc-south-korea-itt/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/mediterranean-games-itt/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-ethiopia/2022
Total brut: 246 | Uniques: 177 | Manquantes: 1
❌ https://www.procyclingstats.com/race/five-rings-of-moscow/2022/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/five-rings-of-moscow/2022/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/five-rings-of-moscow/2022/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/five-rings-of-moscow/2022/stage-4: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/five-rings-of-moscow/2022/stage-5: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/trans-himalaya-cycling-race/2022/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/trans-himalaya-cycling-race/2022/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/trans-himalaya-cycling-ra

In [24]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2022&month=7&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour juillet 2022")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[7] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(7)
driver.quit()

✅ 57 URLs trouvées pour juillet 2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and wi

✅ https://www.procyclingstats.com/race/tour-de-france/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-south-korea/2022


...

✅ https://www.procyclingstats.com/race/grand-prix-yahyali/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-hong-kong/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-paraguay/2022
Total brut: 132 | Uniques: 57 | Manquantes: 0
❌ https://www.procyclingstats.com/race/tour-of-austria/2022/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-austria/2022/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-austria/2022/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-austria/2022/stage-4: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-austria/2022/stage-5: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/chrono-kristin-armstrong/2022/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-de-delta/2022/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/settimana-ciclistica-italiana/2022/stage-1: single positional indexer 

In [25]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2022&month=8&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour aout 2022")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[8] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(8)
driver.quit()

✅ 44 URLs trouvées pour aout 2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and wi

✅ https://www.procyclingstats.com/race/vuelta-a-burgos/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and wi

✅ https://www.procyclingstats.com/race/czech-tour/2022


...

✅ https://www.procyclingstats.com/race/1st-ljubljana-zagreb/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and wi

✅ https://www.procyclingstats.com/race/renewi-tour/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and wi

✅ https://www.procyclingstats.com/race/tour-of-silk-road-hero-cities/2022
Total brut: 136 | Uniques: 44 | Manquantes: 0
❌ https://www.procyclingstats.com/race/tour-of-kosovo/2022/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-kosovo/2022/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-kosovo/2022/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-do-rio/2022/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-do-rio/2022/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-do-rio/2022/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-do-rio/2022/stage-4: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-do-rio/2022/stage-5: single positional indexer is out-of-bounds
❌ https

In [26]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2022&month=9&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour septembre 2022")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[9] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(9)
driver.quit()

✅ 46 URLs trouvées pour septembre 2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and wi

✅ https://www.procyclingstats.com/race/giro-della-regione-friuli-venezia-giulia/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/hafjell-gp/2022


...

✅ https://www.procyclingstats.com/race/tour-of-croatia/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/coppa-agostoni/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and wi

✅ https://www.procyclingstats.com/race/international-azerbaijan-tour/2022
Total brut: 103 | Uniques: 46 | Manquantes: 0
❌ https://www.procyclingstats.com/race/hafjell-gp/2022/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/subway-3-stage-race/2022/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/subway-3-stage-race/2022/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-singkarak/2022/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-singkarak/2022/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-singkarak/2022/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-singkarak/2022/stage-4: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-singkarak/2022/stage-5: single position

In [27]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2022&month=10&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour octobre 2022")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[10] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(10)
driver.quit()

✅ 39 URLs trouvées pour octobre 2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/giro-dell-emilia/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/tour-of-montenegro/2022


...

✅ https://www.procyclingstats.com/race/elite-road-caribbean-championships-itt/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and wi

✅ https://www.procyclingstats.com/race/vuelta-a-guatemala/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/elite-road-caribbean-championships-road-race/2022
Total brut: 81 | Uniques: 39 | Manquantes: 0
❌ https://www.procyclingstats.com/race/tour-of-montenegro/2022/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-montenegro/2022/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-montenegro/2022/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/grand-prix-velo-erciyes-me/2022/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/gp-bruno-beghelli/2022/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/international-syrian-tour/2022/stage-5: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-binzhou/2022/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-binzh

In [28]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2022&month=11&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour novembre 2022")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[11] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(11)
driver.quit()

✅ 13 URLs trouvées pour novembre 2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/gp-sakia-el-hamra/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/gp-oued-eddahab/2022


...

✅ https://www.procyclingstats.com/race/tour-de-okinawa/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and wi

✅ https://www.procyclingstats.com/race/vuelta-ciclista-a-chiriqui-panama/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/grand-prix-municipio-de-david-chiriqui-panama/2022
Total brut: 33 | Uniques: 13 | Manquantes: 0
❌ https://www.procyclingstats.com/race/tour-du-faso/2022/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-du-faso/2022/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-du-faso/2022/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-du-faso/2022/stage-4: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-du-faso/2022/stage-5: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-du-faso/2022/stage-6: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-du-faso/2022/stage-7: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-du-faso/2022/stage-8: single positional indexer is out-of-

In [29]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2022&month=12&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour decembre 2022")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[12] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(12)
driver.quit()

✅ 4 URLs trouvées pour decembre 2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and wi

✅ https://www.procyclingstats.com/race/vuelta-internacional-a-costa-rica/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/arab-road-cycling-championships-road-race/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-china-itt/2022


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_672/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-china/2022
Total brut: 13 | Uniques: 4 | Manquantes: 0
✅ Mois 12: 13 fichiers sauvegardés
